# Phase 1: Document Analysis & JSON Extraction

Goal of this notebook is to analyze a document and extract key attributes 
that help to categorize and manage it automatically.

The LLM is prompted to return a structured JSON response containing metadata such as:
- Document type (invoice, contract, letter, ...)
- Sender / issuer
- Date
- Amount (if applicable)
- Status (open, paid, expired, ...)

Structured output is essential for Phase 3, where the system will use these 
attributes to autonomously decide where to move a file.

In [28]:
from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path
from typing import Optional, List
from pydantic import BaseModel, Field
from datetime import timedelta
import dateparser
import pdfplumber
import os

# Configuration
load_dotenv()
client = OpenAI(api_key=os.getenv("LM_STUDIO_API_KEY"),
base_url=os.getenv("LM_STUDIO_BASE_URL"))
LLM_MODEL = str(os.getenv("LM_STUDIO_MODEL"))
if not LLM_MODEL:
    raise ValueError("LM_STUDIO_MODEL not configured. Check your .env file.")

data_dir = Path.cwd().parent / "data" / "samples"

In [29]:
class DocumentMetadata(BaseModel):
    document_type: str           # invoice, contract, letter, ...
    doc_lang: str # German, English, Frensh, Greek,...
    issuer: str                  # Wer hat das Dokument ausgestellt?
    recipient: List[str]     # An wen geht es?
    date: Optional[str]          # Ausstellungsdatum
    due_date: Optional[str]      # Fälligkeitsdatum (bei Rechnungen)
    amount: Optional[float]      # Betrag (wenn vorhanden)
    currency: Optional[str]      # EUR, USD, ...
    status: Optional[str]        # open, paid, overdue, ...
    document_id: Optional[str] = Field(description="Invoice number, e.g. RE-2026-001")
    customer_id: Optional[str] = Field(description="Customer number, e.g. Kd-Nr. 1010")
    payment_terms_days: Optional[int]  # z.B. 14
    iban: Optional[str]
    bic: Optional[str]

In [30]:
test_files = ["GB.pdf", "amazon.pdf", "invoice_001_open.pdf"]

for filename in test_files:
    pdf_path = data_dir / filename

    # pdfplumber
    with pdfplumber.open(pdf_path) as pdf:
        text_pdfplumber = pdf.pages[0].extract_text()

    document_text = text_pdfplumber

    completion = client.chat.completions.create(
        model=LLM_MODEL,  # exakter Name aus LM Studio
        messages=[
            {"role": "system", "content": """
            Du bist ein Buchhaltungs- und Ablage-Assistent.

    Aufgabe:
    Extrahiere die angeforderten Felder ausschließlich aus dem vorliegenden Dokument.

    Felddefinitionen:
    - issuer: Absender oder ausstellende Firma
    - receipient: Adressat
    - doc_lang: Englische Bezeichnung der Sprache, in der das Dokument verfasst ist                  
    - document_type: Englische Bezeichnung des Dokument-Typs: "Rechnung" -> "Invoice"         
    - document_id: Nur die Rechnungsnummer. Erkennbar an Begriffen wie "Rech.-Nr.", "Rechnungsnummer", "Invoice No." Beispiel: GB-2026/001
    - customer_id: Nur die Kundennummer. Erkennbar an Begriffen wie "Kd-Nr.", "Kundennummer", "Customer No." Beispiel: 1010
    - date: Ausstellungsdatum des Dokuments. Erkennbar an Begriffen wie "Datum:", "Rechnungsdatum", "Date".
    - due_date: Explizit genanntes Fälligkeitsdatum Erkennbar an Begriffen wie 
    "Fällig bis", "Due date", "Zahlbar bis". Das Bestelldatum ist KEIN Fälligkeitsdatum.
    - iban: Bankverbindung IBAN. Erkennbar an "IBAN:" gefolgt von einem Kontocode.
    - bic: BIC/SWIFT Code der Bank. Erkennbar an "BIC:" oder "SWIFT:".                  
            
    Regeln:
    - Extrahiere nur Informationen die explizit im Dokument stehen
    - Erfinde oder schlussfolgere keine Werte
    - Wenn ein Feld nicht eindeutig erkennbar ist, gib null zurück
    - Der Status (z.B. open, paid, overdue) darf nur gesetzt werden wenn er 
    ausdrücklich im Dokument erwähnt wird
    - Beträge werden als Dezimalzahl mit zwei Nachkommastellen zurückgegeben"""},
            {"role": "user", "content": f"Ich habe folgendes Dokument bekommen\n {document_text}"}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "DocumentMetadata",
                "schema": DocumentMetadata.model_json_schema()
            }
        },
    )
    content = completion.choices[0].message.content
    if not content:
        raise ValueError("Model returned empty response")

    result = DocumentMetadata.model_validate_json(content)
    print(f"Raw due_date: {result.due_date}")

    for field in DocumentMetadata.model_fields:
        if getattr(result, field) == 'null':
            setattr(result, field, None)
    
    if result.due_date is None and result.date is not None and result.payment_terms_days is not None and result.doc_lang is not None:
        if result.doc_lang != "English":
            doc_date = dateparser.parse(result.date, languages=["de", "en"], settings={"DATE_ORDER": "DMY"})
        else:
            doc_date = dateparser.parse(result.date, languages=["de", "en"], settings={"DATE_ORDER": "YMD"})
        if doc_date:
            result.due_date = (doc_date + timedelta(days=result.payment_terms_days)).strftime("%Y-%m-%d")
        else:    
            print(f"⚠️ Datumsformat nicht erkannt: {result.date}")
    print(f"\n=== {filename} ===")
    print(result)

Raw due_date: None

=== GB.pdf ===
document_type='Invoice' doc_lang='German' issuer='Gut Beraten Lohnsteuerhilfe e.V.' recipient=['Julia Wurm', 'Jesco Wurm'] date='3.1.2026' due_date='2026-01-17' amount=100.0 currency='EUR' status=None document_id='GB-2026/001' customer_id='1010' payment_terms_days=14 iban='DE24 1001 1001 2657 1068 02' bic='NTSBDEB1XXX'
Raw due_date: None

=== amazon.pdf ===
document_type='Invoice' doc_lang='German' issuer='Amazon EU S.à r.l., Niederlassung Deutschland' recipient=['JESCO ALEXANDER WURM'] date='18 Februar 2026' due_date=None amount=54.9 currency='EUR' status=None document_id='DE6LLJ5FAEUI' customer_id=None payment_terms_days=None iban=None bic=None
Raw due_date: 2026-03-15

=== invoice_001_open.pdf ===
document_type='Invoice' doc_lang='English' issuer='TechSolutions GmbH' recipient=['Sample Client AG', 'Attn: Accounts Payable'] date='2026-02-15' due_date='2026-03-15' amount=2332.4 currency='EUR' status='OPEN' document_id='INV-2026-0042' customer_id=None

## Prompt Engineering: Null over guessing

The first prompt version produced hallucinated values for fields that were 
not present in the document (e.g. status='paid', due_date=date).

Adding an explicit rule — "if a field is not clearly present, return null" — 
eliminated all hallucinations. This is a critical pattern for reliable 
structured extraction.

## Performance Note
Token usage per extraction: ~880 tokens (prompt) + ~145 tokens (response).
For batch processing of large document volumes, prompt optimization 
will be relevant in later phases.